In [ ]:
import gymnasium as gym

import numpy as np
import polars as pl

from collections import defaultdict

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

from market import SingleMarket, MultiMarket

In [ ]:

class RandomSingleSeller:
    def __init__(
        self,
    ):
        pass

    def get_action(self, env, state):
        action = env.action_space.sample()
        return action

    def update(self, state, action, reward, next_state, next_action, info):
        pass


In [ ]:
class FixedSingleSeller:
    def __init__(
        self
    ):
        pass

    def get_action(self, env, state):
        return env.action_space.n//2

    def update(self, state, action, reward, next_state, next_action, info):
        pass


In [ ]:
class LearningSingleSeller:
    def __init__(
        self,
    ):
        self.value_function = defaultdict(lambda: 0)
        self.epsilon = 0.1
        self.alpha = 0.10
        self.learning_method = "sarsa"
        self.learning_rate = "decaying-epsilon"
        self.t = 0

    def _get_epsilon(self):
        if self.learning_rate == "decaying-epsilon":
            return 1 / (self.t + 1)
        elif self.learning_rate == "constant":
            return 0.1

    def get_action(self, env, state):
        epsilon = self._get_epsilon()
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(
                [self.value_function[state, i] for i in range(env.action_space.n)]
            )
        return int(action)

    def update(self, state, action, reward, next_state, next_action, info):
        if self.learning_method == "sarsa":
            self.value_function[state, action] += self.alpha * (
                reward
                + self.value_function[next_state, next_action]
                - self.value_function[state, action]
            )
        self.t += 1


In [ ]:
def generate_episodes(env, agent, n_episodes=2):
    sequence = []
    for r in range(n_episodes):
        terminated = False

        # initialize:
        state = env.reset()
        action = agent.get_action(env, state)

        while not terminated:
            # update loop
            next_state, reward, terminated, info = env.step(action)
            next_action = agent.get_action(env, next_state)
            agent.update(state, action, reward, next_state, next_action, info)
            action, state = next_action, next_state

            sequence.append((state, action, reward, r, info))

    episodes = pl.DataFrame(
        sequence,
        schema=["state", "action", "reward", "episode", "info"],
        orient="row",
    )

    episodes = episodes.with_columns(
        pl.col("state").list.get(0).alias("t"),
        pl.col("state").list.get(1).alias("stock"),
    )

    return episodes


In [ ]:
env = SingleMarket(max_price=10, n_periods=10)
agent = RandomSingleSeller()

episodes = generate_episodes(env, agent, n_episodes=2)
episodes

In [ ]:
env = SingleMarket(max_price=10, n_periods=10)
agent = FixedSingleSeller()

episodes = generate_episodes(env, agent, n_episodes=1000)
episodes

px.line(
    episodes.group_by("episode")
    .agg(pl.sum("reward").alias("total_reward"))
    .sort("episode"),
    x="episode",
    y="total_reward",
)

In [ ]:
env = SingleMarket(max_price=4, n_periods=10)
agent = LearningSingleSeller()

episodes = generate_episodes(
    env,
    agent,
    n_episodes=1000,
)
episodes


In [ ]:
value_function = dict(agent.value_function)

In [ ]:
list(value_function.keys())[0]

In [ ]:
px.line(
    episodes.group_by("episode").agg(pl.sum("reward").alias("total_reward")).sort("episode"),
    x="episode", y="total_reward",
)